# Стоимость квартир в Москве

**Задача:** предсказать цену квартиры `Price` для 10 127 объектов из `test.csv`.
**Метрика:** RMSE между логарифмами прогноза и факта, то есть ошибка в «процентах» цены.

**Решение коротко:**
- модель предсказывает логарифм цены квадратного метра;
- итог — простое среднее четырёх разных моделей деревьев: LightGBM, два варианта XGBoost и ExtraTrees;
- признаки: 10 исходных, плюс счётчики квартир в «доме» и в районе с той же этажностью, плюс этажность дома как категория;
- оценка качества: 5-кратная кросс-валидация;
- прогноз для test — среднее моделей, обученных на фолдах.

**Результат:** CV 0.1700 против 0.1796 у baseline. Тетрадь выполняется с нуля примерно за 5 минут.

In [1]:
import os, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import shap
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

SEED = 42
N_JOBS = os.cpu_count()
START = time.time()
pd.set_option("display.width", 140)

## 1. Данные

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
FEATURES = ["Room", "Size", "Balcony", "Floor", "FloorsTotal", "Walls", "Age", "Lift", "District", "Okrug"]
CAT_COLS = ["Balcony", "Walls", "Age", "Lift", "District", "Okrug"]
n = len(train)
print(train.shape, test.shape)
pd.DataFrame({"пропуски train": train[FEATURES].isna().mean(), "пропуски test": test[FEATURES].isna().mean()}).round(3).T

(40508, 12) (10127, 11)


,Room,Size,Balcony,Floor,FloorsTotal,Walls,Age,Lift,District,Okrug
пропуски train,0.0,0.0,0.315,0.0,0.0,0.145,0.013,0.233,0.009,0.009
пропуски test,0.0,0.0,0.319,0.0,0.0,0.149,0.014,0.225,0.010,0.010


## 2. Что важно в данных

**Таргет.** Цена от 1,7 млн до 2,3 млрд ₽. Метрика считается в логарифмах, поэтому моделируем логарифм.
Удобнее предсказывать логарифм цены **квадратного метра**: его разброс почти вдвое меньше, а площадь возвращается в прогноз умножением.

In [3]:
y = np.log(train.Price)                      # то, что оценивает метрика
t = np.log(train.Price / train.Size)        # то, что предсказывают модели
log_size = np.log(train.Size).values
print(f"std log(цены) = {y.std():.3f}, std log(цены м²) = {t.std():.3f}")

std log(цены) = 0.890, std log(цены м²) = 0.501


**Пропуски несут информацию.** Например, квартиры без указанного материала стен в среднем дороже соседей по району.
Поэтому пропуски не заполняем, а делаем отдельной категорией `NA`.

In [4]:
t_rel = t - t.groupby(train.District).transform("mean")    # цена м² относительно своего района
pd.Series({c: t_rel[train[c].isna()].mean() - t_rel[train[c].notna()].mean()
           for c in ["Balcony", "Walls", "Lift", "Age"]}, name="разница log(цены м²): пропуск − нет").round(3)

Balcony    0.057
Walls      0.069
Lift      -0.030
Age       -0.048
Name: разница log(цены м²): пропуск − нет, dtype: float64

**Train и test неотличимы.** Классификатор не может понять, из какой выборки строка: AUC ≈ 0.5.
Значит, разбиение случайное, и обычная случайная кросс-валидация честно имитирует test.

In [5]:
def to_model_frame(df):
    X = df[FEATURES].copy()
    for c in CAT_COLS:
        X[c] = X[c].fillna("NA").astype("category")
    return X

both = to_model_frame(pd.concat([train, test], ignore_index=True))
is_test = np.r_[np.zeros(n), np.ones(len(test))]
p = cross_val_predict(lgb.LGBMClassifier(n_estimators=200, verbose=-1), both, is_test,
                      cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
print(f"adversarial AUC = {roc_auc_score(is_test, p):.3f}")

adversarial AUC = 0.493


**«Дом».** Квартиры с одинаковыми районом, этажностью, материалом стен и типом жилья почти наверняка стоят в одном здании.
Внутри такой группы цена м² разбросана заметно меньше, чем внутри района. Поэтому моделям нужны глубокие деревья с маленькими листьями: они должны уметь выделять отдельные дома.

In [6]:
HOUSE = ["District", "FloorsTotal", "Walls", "Age"]
def make_key(df, cols):
    k = df[cols[0]].astype("object").fillna("NA").astype(str)
    for c in cols[1:]:
        k = k + "|" + df[c].astype("object").fillna("NA").astype(str)
    return k.to_numpy()

house = make_key(train, HOUSE)
big = pd.Series(house).map(pd.Series(house).value_counts()).values >= 5
within_house = t - t.groupby(house).transform("mean")
print(f"std цены м² для домов с ≥5 квартирами: внутри района {t_rel[big].std():.3f}, внутри дома {within_house[big].std():.3f}")

std цены м² для домов с ≥5 квартирами: внутри района 0.292, внутри дома 0.187


**Выбросы не удаляем.** Самые дорогие объекты за м² — реальная элитка: посёлок Сокол, Хамовники, Покровское-Стрешнево. Такие же квартиры есть в test.
Удаление «аномальных» строк из обучения на кросс-валидации только ухудшало результат.

**Порядок Id не используем.** Строки идут не случайно: соседние по Id квартиры часто из одного дома. Это артефакт подготовки файлов, а не свойство квартиры, поэтому такой признак не строим.

## 3. Валидация

5-кратная кросс-валидация с фиксированным разбиением: все варианты сравниваются на одних и тех же фолдах.

Каждая модель:
- выдаёт **out-of-fold** прогноз: каждая строка train предсказана моделью, которая её не видела;
- выдаёт прогноз для test: среднее пяти фолдовых моделей.

Отдельно на всём train модели не переобучаем: это экономит время, а среднее пяти моделей ещё и стабильнее одной.

In [7]:
FOLDS = list(KFold(5, shuffle=True, random_state=SEED).split(train))

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def cv(make_model, X, X_test, target=t, encode=None):
    """OOF-прогноз и прогноз test (среднее фолдовых моделей). encode(A, [B, T], target_A) — кодировка внутри фолда."""
    oof, pred_test, models = np.zeros(len(X)), np.zeros(len(X_test)), []
    for tr_idx, va_idx in FOLDS:
        A, B, T = X.iloc[tr_idx], X.iloc[va_idx], X_test
        if encode is not None:
            A, (B, T) = encode(A, [B, T], target.iloc[tr_idx])
        m = make_model().fit(A, target.iloc[tr_idx])
        oof[va_idx] = m.predict(B)
        pred_test += m.predict(T) / len(FOLDS)
        models.append(m)
    return oof, pred_test, models

X, X_test = to_model_frame(train), to_model_frame(test)
for c in CAT_COLS:   # в test те же категории, что и в train
    X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

## 4. Выбор таргета

Одна и та же модель LightGBM со стандартными настройками, меняется только таргет. Категории LightGBM обрабатывает нативно: это лучше one-hot и порядковых кодов (0.178 против 0.184).

In [8]:
base_lgb = lambda: lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, num_leaves=63, subsample=0.8,
                                     subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1, n_jobs=N_JOBS)
oof_price, _, _ = cv(base_lgb, X, X_test, target=y)
oof_ppm, _, _ = cv(base_lgb, X, X_test)
pd.Series({"baseline: таргет log(цены)": rmse(y, oof_price),
           "таргет log(цены м²)": rmse(y, oof_ppm + log_size)}, name="CV RMSE").round(4)

baseline: таргет log(цены)    0.1796
таргет log(цены м²)           0.1774
Name: CV RMSE, dtype: float64

## 5. Признаки

Три представления одних и тех же данных. На них учатся разные модели, поэтому модели ошибаются по-разному.

1. **Исходные 10 признаков.**
2. **Счётчики без таргета**, посчитанные по train и test вместе:
   - `house_cnt` — сколько квартир в «доме»;
   - `dft_cnt` — сколько квартир в районе с той же этажностью дома.

   Много одинаковых объявлений означает массовую застройку или новостройку на старте продаж, одно объявление — штучный дом. Цены в счётчиках не участвуют.
3. **Этажность дома как категория.** Типовые серии (5, 9, 12, 14, 17, 22 этажа) — это разные типы зданий, а не точки на числовой оси. Категория позволяет одним разбиением выделить, например, «все 9- и 12-этажки».

Для ExtraTrees категории кодируются рангом средней цены м² внутри обучающей части фолда. Это тот же приём, которым LightGBM упорядочивает категории внутри себя.

In [9]:
full = pd.concat([train, test], ignore_index=True)
count = lambda key: pd.Series(key).map(pd.Series(key).value_counts()).to_numpy()

X_cnt, X_cnt_test = X.copy(), X_test.copy()
for name, cols in [("house_cnt", HOUSE), ("dft_cnt", ["District", "FloorsTotal"])]:
    c = count(make_key(full, cols))
    X_cnt[name], X_cnt_test[name] = c[:n], c[n:]

floors_cats = sorted(train.FloorsTotal.unique())
X_ft, X_ft_test = X.copy(), X_test.copy()
X_ft["FloorsTotal"] = pd.Categorical(X_ft.FloorsTotal, categories=floors_cats)
X_ft_test["FloorsTotal"] = pd.Categorical(X_ft_test.FloorsTotal.where(X_ft_test.FloorsTotal.isin(floors_cats)),
                                          categories=floors_cats)   # этажность, которой нет в train, -> пропуск

R = full[FEATURES].copy()                      # кадр для ExtraTrees
for c in CAT_COLS:
    R[c] = R[c].astype("object").fillna("NA")
R["FT_cat"] = R.FloorsTotal.astype(str)
R, R_test = R.iloc[:n].reset_index(drop=True), R.iloc[n:].reset_index(drop=True)

def rank_encode(A, others, target_A):
    """Категория -> ранг средней цены м² в обучающей части фолда (неизвестная категория -> -1)."""
    A, others = A.copy(), [o.copy() for o in others]
    for c in CAT_COLS + ["FT_cat"]:
        r = target_A.groupby(A[c].values).mean().rank().to_dict()
        A[c] = A[c].map(r).astype(float)
        for o in others:
            o[c] = o[c].map(r).fillna(-1).astype(float)
    return A, others

## 6. Модели

Все четыре модели — деревья с крупными листьями, способные выделять отдельные дома. Различаются они алгоритмом и представлением данных.

| Модель | Данные | Чем отличается |
|---|---|---|
| LightGBM | исходные | 127 листьев, `min_child_samples=5` (дом из 5 квартир может стать листом), L2-регуляризация 5, все признаки в каждом дереве (их всего 10) |
| XGBoost (leaf-wise) | + счётчики | Деревья растут по листьям, как в LightGBM, но с другой обработкой категорий и регуляризацией |
| XGBoost (leaf-wise) | этажность — категория | Другое разбиение по этажности |
| ExtraTrees | ранги категорий | Случайные пороги разбиений и полностью иная ошибка; слабее сама по себе, но полезна в среднем |

Состав выбран жадно по out-of-fold прогнозам из ~40 проверенных вариантов (подробности в `REPORT.md`): модель добавлялась, только если среднее с ней становилось точнее. Веса равные.

In [10]:
lgb_params = dict(n_estimators=3000, learning_rate=0.02, num_leaves=127, min_child_samples=5, reg_lambda=5,
                  subsample=0.8, subsample_freq=1, colsample_bytree=1.0, random_state=SEED, verbose=-1, n_jobs=N_JOBS)
xgb_params = dict(n_estimators=1500, learning_rate=0.03, grow_policy="lossguide", max_leaves=127, max_depth=0,
                  subsample=0.8, colsample_bytree=0.8, enable_categorical=True, tree_method="hist",
                  max_cat_to_onehot=1, random_state=SEED, n_jobs=N_JOBS)

results = {}
for name, make_model, D, D_test, enc in [
    ("LightGBM", lambda: lgb.LGBMRegressor(**lgb_params), X, X_test, None),
    ("XGBoost + счётчики", lambda: xgb.XGBRegressor(**{**xgb_params, "learning_rate": 0.02, "n_estimators": 2250}), X_cnt, X_cnt_test, None),
    ("XGBoost, этажность-категория", lambda: xgb.XGBRegressor(**xgb_params), X_ft, X_ft_test, None),
    ("ExtraTrees", lambda: ExtraTreesRegressor(n_estimators=200, max_features=1.0, n_jobs=N_JOBS, random_state=SEED), R, R_test, rank_encode),
]:
    t0 = time.time()
    oof, pred, models = cv(make_model, D, D_test, encode=enc)
    results[name] = {"oof": oof + log_size, "test": pred, "models": models}
    print(f"{name:30s} CV {rmse(y, results[name]['oof']):.4f}   {time.time() - t0:5.0f} с")

LightGBM                       CV 0.1736      59 с


XGBoost + счётчики             CV 0.1726      62 с


XGBoost, этажность-категория   CV 0.1733      43 с


ExtraTrees                     CV 0.1796      25 с


In [11]:
oof_avg = np.mean([r["oof"] for r in results.values()], axis=0)
oof_names = list(results)
resid_corr = pd.DataFrame({k: results[k]["oof"] - y for k in oof_names}).corr().round(3)
print(f"Среднее четырёх моделей: CV {rmse(y, oof_avg):.4f}")
print("По фолдам:", [round(rmse(y.values[va], oof_avg[va]), 4) for _, va in FOLDS])
resid_corr

Среднее четырёх моделей: CV 0.1700
По фолдам: [0.1687, 0.1749, 0.1667, 0.1708, 0.169]


,LightGBM,XGBoost + счётчики,"XGBoost, этажность-категория",ExtraTrees
LightGBM,1.000,0.963,0.946,0.898
XGBoost + счётчики,0.963,1.000,0.964,0.907
"XGBoost, этажность-категория",0.946,0.964,1.000,0.898
ExtraTrees,0.898,0.907,0.898,1.000


Ошибки моделей скоррелированы на 0.90–0.96, а не на 1. Поэтому среднее заметно точнее лучшей отдельной модели.

## 7. Что выучила модель

Важность признаков в LightGBM (доля разбиений) и SHAP — насколько признак в среднем сдвигает прогноз. Берём модель первого фолда.

In [12]:
lgb_model = results["LightGBM"]["models"][0]
sample = X.sample(2000, random_state=SEED)
shap_values = shap.TreeExplainer(lgb_model).shap_values(sample)
pd.DataFrame({"доля разбиений": lgb_model.feature_importances_ / lgb_model.feature_importances_.sum(),
              "средний |SHAP|": np.abs(shap_values).mean(axis=0)},
             index=X.columns).sort_values("средний |SHAP|", ascending=False).round(3)

,доля разбиений,средний |SHAP|
District,0.090,0.285
Size,0.338,0.081
Walls,0.022,0.060
FloorsTotal,0.197,0.032
Age,0.024,0.024
Floor,0.182,0.024
Room,0.036,0.014
Balcony,0.057,0.012
Lift,0.052,0.009
Okrug,0.003,0.001


Главные признаки — район и площадь, затем материал стен и этажность дома.

## 8. Где модель ошибается

In [13]:
err = pd.DataFrame({"Okrug": train.Okrug.fillna("NA"),
                    "сегмент": pd.qcut(train.Price, 5, labels=["дешёвые", "ниже ср.", "средние", "выше ср.", "дорогие"]),
                    "sq": (oof_avg - y) ** 2})
display(err.groupby("Okrug").sq.mean().pow(0.5).round(3).sort_values().to_frame("RMSE").T)
display(err.groupby("сегмент", observed=True).sq.mean().pow(0.5).round(3).to_frame("RMSE").T)

Okrug,ЮВАО,ВАО,ЮАО,СВАО,ЮЗАО,СЗАО,САО,ЗАО,NA,ЦАО
RMSE,0.111,0.122,0.129,0.132,0.142,0.142,0.143,0.202,0.221,0.274


сегмент,дешёвые,ниже ср.,средние,выше ср.,дорогие
RMSE,0.119,0.12,0.135,0.168,0.264


Хуже всего модель работает в ЦАО и в дорогом сегменте: там цена зависит от того, чего нет в данных (конкретный дом, вид, отделка).

## 9. Прогноз для test

Прогноз каждой модели — среднее её пяти фолдовых моделей. Четыре модели усредняются в логарифмах, затем переводятся обратно в цену: `Price = exp(log(цены м²)) × Size`.

In [14]:
pred_t = np.mean([r["test"] for r in results.values()], axis=0)
submission = pd.DataFrame({"Id": test.Id, "Price": np.exp(pred_t) * test.Size})

assert len(submission) == 10127 and submission.Id.equals(test.Id)
assert submission.Price.notna().all() and (submission.Price > 0).all()
submission.to_csv("submissions/submission.csv", index=False)
submission.describe().round(0)

,Id,Price
count,10127.0,10127.0
mean,45572.0,23042822.0
std,2924.0,41462267.0
min,40509.0,1723738.0
25%,43040.0,7137147.0
50%,45572.0,10367684.0
75%,48104.0,19378205.0
max,50635.0,688190817.0


In [15]:
print(f"Итоговый CV RMSE: {rmse(y, oof_avg):.4f}")
print(f"Время выполнения тетради: {(time.time() - START) / 60:.1f} мин")

Итоговый CV RMSE: 0.1700
Время выполнения тетради: 4.7 мин
